# Practical Exam: Hotel Operations

LuxurStay Hotels is a major, international chain of hotels. They offer hotels for both business and leisure travellers in major cities across the world. The chain prides themselves on the level of customer service that they offer. 

However, the management has been receiving complaints about slow room service in some hotel branches. As these complaints are impacting the customer satisfaction rates, it has become a serious issue. Recent data shows that customer satisfaction has dropped from the 4.5 rating that they expect. 

You are working with the Head of Operations to identify possible causes and hotel branches with the worst problems. 

## Data

The following schema diagram shows the tables available. You have only been provided with data where customers provided a feedback rating.

![hotel_operations](hotel_operations.png)

# Task 1

Before you can start any analysis, you need to confirm that the data is accurate and reflects what you expect to see. 

It is known that there are some issues with the `branch` table, and the data team have provided the following data description. 

Write a query to return data matching this description. You must match all column names and description criteria.

| Column Name | Criteria                                                |
|-------------|---------------------------------------------------------|
|id | Nominal. The unique identifier of the hotel. </br>Missing values are not possible due to the database structure.|
| location | Nominal. The location of the particular hotel. One of four possible values, 'EMEA', 'NA', 'LATAM' and 'APAC'. </br>Missing values should be replaced with “Unknown”. |
| total_rooms | Discrete. The total number of rooms in the hotel. Must be a positive integer between 1 and 400. </br>Missing values should be replaced with the default number of rooms, 100. |
| staff_count | Discrete. The number of staff employeed in the hotel service department. </br>Missing values should be replaced with the total_rooms multiplied by 1.5. |
| opening_date | Discrete. The year in which the hotel opened. This can be any value between 2000 and 2023. </br>Missing values should be replaced with 2023. |
| target_guests | Nominal. The primary type of guest that is expected to use the hotel. Can be one of 'Leisure' or 'Business'. </br>Missing values should be replaced with 'Leisure'. |

In [ ]:
#Task 1 query

#"branch" table has 100 total rows
    #'id' INT, 'location' text, 'total_rooms' INT, 'staff_count' INT, 'opening_date' text, 'target_guests' text
#column data types (use PG_TYPEOF(col)):
    #opening_date can be changed from text to integer

#Check data types of columns
	--SELECT *
	--FROM INFORMATION_SCHEMA.columns
	--WHERE table_name = 'branch';

#Investigate value counts of columns
SELECT target_guests, COUNT(target_guests), SUM(COUNT(target_guests)) OVER() AS sum
FROM branch
GROUP BY target_guests
ORDER BY target_guests;

SELECT target_guests
FROM branch
WHERE target_guests IS NULL;

#Columns to fix: 'total_rooms' (missing values), 'opening_date' (missing values, data type), 'target_guests' (typos/inconsistencies in values)
#Columns that are good: 'id', 'location', 'staff_count'


#Adjusting columns as necessary
    #Column 'total_rooms': Replace null values with 100
SELECT total_rooms, COALESCE(total_rooms, 100) AS new_tot_rooms
FROM branch;
    #Column 'opening_date': Change data types of values from text to integers. Replace missing values ("-") with 2023.
SELECT REPLACE(opening_date, '-', '2023')::INT AS new_opening_date
FROM branch;
    #Column 'target_guests': Correct inconsistencies in values ("B.", "Buisness")
SELECT target_guests, 
    CASE WHEN target_guests = 'B.' THEN 'Business' 
        WHEN target_guests = 'Busniess' THEN 'Business' 
        WHEN target_guests = 'Business' THEN 'Business'
        ELSE 'Leisure' END AS new_target_guests
FROM branch;


#Check column adjustments
WITH task1_adj AS (
    SELECT id, location, 
        COALESCE(total_rooms, 100) AS new_total_rooms, 
        staff_count, 
        REPLACE(opening_date, '-', '2023')::INT AS new_opening_date, 
        CASE WHEN target_guests = 'B.' THEN 'Business' 
            WHEN target_guests = 'Busniess' THEN 'Business' 
            WHEN target_guests = 'Business' THEN 'Business'
            ELSE 'Leisure' END AS new_target_guests
    FROM branch
)
SELECT *
FROM task1_adj;
#GOOD

#Output final query
SELECT id, location, 
    COALESCE(total_rooms, 100) AS total_rooms, 
    staff_count, 
    REPLACE(opening_date, '-', '2023')::INT AS opening_date, 
    CASE WHEN target_guests = 'B.' THEN 'Business' 
        WHEN target_guests = 'Busniess' THEN 'Business' 
        WHEN target_guests = 'Business' THEN 'Business'
        ELSE 'Leisure' END AS target_guests
FROM branch;
    #returns 100 rows


# Task 2

The Head of Operations wants to know whether there is a difference in time taken to respond to a customer request in each hotel. They already know that different services take different lengths of time. 

Calculate the average and maximum duration for each branch and service. Your output should include the columns `service_id`, `branch_id`, `avg_time_taken` and `max_time_taken`. Values should be rounded to two decimal places where appropriate. 

In [ ]:
#Task 2 query

#in "request" table, are 17,682 rows

SELECT service_id, branch_id, ROUND(AVG(time_taken), 2) AS avg_time_taken, MAX(time_taken) AS max_time_taken
FROM request
GROUP BY service_id, branch_id;
    #385 rows


# Task 3

The management team want to target improvements in `Meal` and `Laundry` service in Europe (`EMEA`) and Latin America (`LATAM`). 

Write a query to return the `description` of the service, the `id` and `location` of the branch, the id of the request as `request_id` and the `rating` for the services and locations of interest to the management team. 

Use the original branch table, not the output of task 1. 

In [ ]:
#Task 3 query

#"service" table has 4 rows

#Need to apply filters to: 'description' in the "service" table, & 'location' in the "branch" table

SELECT description, B.id, location, R.id AS request_id, rating
FROM branch AS B
INNER JOIN request AS R
ON B.id = R.branch_id
INNER JOIN service AS S
ON R.service_id = S.id
WHERE location IN ('EMEA', 'LATAM') AND description IN ('Meal', 'Laundry');

#OR

SELECT description, B.id, location, R.id AS request_id, rating
FROM request AS R
INNER JOIN service AS S
ON R.service_id = S.id
INNER JOIN branch AS B
ON R.branch_id = B.id
WHERE description IN ('Meal', 'Laundry') AND location IN ('EMEA', 'LATAM');

#FWIW, both INNER JOIN & FULL JOIN return 5,047 rows


# Task 4

So that you can take a more detailed look at the lowest performing hotels, you want to get service and branch information where the average rating for the branch and service combination is lower than 4.5 - the target set by management.  

Your query should return the `service_id` and `branch_id`, and the average rating (`avg_rating`), rounded to 2 decimal places.

In [ ]:
#Task 4 query

#Need to apply a filter for which AVG rating per brachn & service is < 4.5
#Round AVG rating to 2 decimals

SELECT service_id, branch_id, ROUND(AVG(rating), 2) AS avg_rating
FROM request
GROUP BY service_id, branch_id
HAVING AVG(rating) < 4.5
ORDER BY avg_rating;
    #returns 215 rows
